# 06 Text Model (Single-Mode)

Predicts `text_fake` (was the caption manipulated?) **from the caption alone**.
Together with the single-mode image model, it forms the late-fusion approach that the multimodal models are compared against: does looking at the image and caption _together_ beat two separate single-mode models?

**What the caption alone can and cannot catch** (single-feature AUCs from `03`):
- `text_attribute`: 1-2 sentiment words flipped. This is detectable from the text: strongest word valence AUC 0.72, VADER neutral share 0.74 (inverted), GPT-2 surprise 0.63.
- `text_swap`: the whole caption replaced with a real caption about the same person. It is mostly undetectable without the image (best: number of people mentioned, 0.69). A weak score here is expected. It is the gap the multimodal models should close, and showing it is part of the result.

**Architecture: a two-level stack**

| Level | Model | Input | Output |
|---|---|---|---|
| 1 | Logistic regression | TF-IDF word 1–2 grams + character 3–5 grams | P(caption manipulated) |
| 1 | Gradient boosting **per word** | Per-word sentiment, GPT-2 surprise, part of speech, position (section 5.1) | P(this word was edited), summarised over the caption's words |
| 1 | Logistic regression | CLIP caption embedding, text only | P(caption manipulated) |
| 2 | Gradient boosting | Level-1 scores + hand-crafted `txt_*` features (`03`) | `text_fake` |

Why this design:
- **TF-IDF n-grams:** Word n-grams pick up specific flipped phrases ("refuse" ↔ "welcome"), character n-grams pick up odd word forms. The vocabulary is learned, so it is fitted inside the model on train only.
- **Word level:** in training we know which words were edited (`fake_text_pos`), so the word classifier learns from the edited word itself rather than the whole caption. At prediction time it only uses the caption, and the caption score is the most suspicious word. It also gives a **grounding** result: is the most suspicious word the one that was edited?
- **CLIP caption embedding:** `05` uses it too, so giving it to both approaches leaves cross-modal information as the only difference. A multimodal gain then cannot be explained by CLIP simply being a stronger text encoder. It is computed from the caption alone, so this model stays single-mode. Section 8 reports results with and without it.
- **Leakage control:** level-1 scores for training rows are **out-of-fold** (5-fold, grouped by article `id`), so level 2 never sees a score from a model that was trained on that row.

**Leakage rules**
- Folds are grouped by article `id`. An original caption and its text_attribute copy differ by 1-2 words, so ungrouped folds would leak.
- The target is `text_fake`, not `is_fake`. Image-only fakes keep their original caption and count as negatives.
- `fake_text_pos` is used **only as the training label** of the word classifier. `fake_cls`, `image_source` and the `image` path are never features.

**Data use:** models are fit on `train_sample`, compared and tuned on `validation`. `test` is evaluated only in the last section, behind `RUN_TEST`, once the final model is chosen.

## 0. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import best_f1_threshold, binary_metrics, detection_rate_by_type
from src.feature_sets import ARTIFACT_FEATURES, feature_columns

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
CLIP_CACHE = PROJECT_ROOT / "data" / "interim" / "clip"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 4012
N_FOLDS = 5
SPLITS = ["train_sample", "validation", "test"]

# Leave False while developing. Set True once, at the end, to report final test results.
RUN_TEST = False

sns.set_theme(style="whitegrid")

In [ ]:
def clip_caption_embedding(split):
    # clip_emb holds 768 image + 768 caption columns; read only the caption half
    path = FEATURES_DIR / f"clip_emb_{split}.parquet"
    cols = [c for c in pq.read_schema(path).names if c.startswith("clip_txt_")]
    return pd.read_parquet(path, columns=["row_id"] + cols)


def load_split(split):
    df = pd.read_parquet(PROCESSED_DIR / f"{split}.parquet")
    df = df.merge(pd.read_parquet(FEATURES_DIR / f"text_{split}.parquet"), on="row_id", how="left")
    df = df.merge(clip_caption_embedding(split), on="row_id", how="left")
    assert df.filter(regex="^(txt|clip_txt)_").notna().all().all(), f"missing features in {split}"
    return df


data = {split: load_split(split) for split in SPLITS}
train, val = data["train_sample"], data["validation"]

TXT = feature_columns(train, "txt_")  # artifact features excluded
CLIP_TXT = [c for c in train.columns if c.startswith("clip_txt_")]
print({split: df.shape for split, df in data.items()})
print(f"{len(TXT)} txt_* features, {len(CLIP_TXT)} CLIP caption dimensions")

## 1. Caption Labels

The caption manipulation mix per split, and how much of each edited caption `fake_text_pos` marks. This decides how the word-level classifier is labelled (section 5.2).

In [ ]:
pd.DataFrame({
    split: df["text_manipulation"].value_counts(normalize=True) * 100 for split, df in data.items()
}).round(2)

In [ ]:
# Share of the caption's words marked as edited, per caption manipulation type
edited = train[train["text_fake"] == 1]
marked_share = edited["fake_text_pos"].str.len() / edited["word_count"]
marked_share.groupby(edited["text_manipulation"]).describe().round(2)

For `text_attribute`, `fake_text_pos` marks ✏️[FILL: e.g. a median of X% of the caption's words, i.e. the one or two flipped words]. For `text_swap`, it marks ✏️[FILL: e.g. a median of Y% of the words]: the whole caption is foreign, so these positions do not point to specific suspicious words. **The word classifier is therefore trained on unchanged and text_attribute captions only.**

## 2. Shared Helpers

**One fold assignment for the whole notebook.** Every article `id` in `train_sample` is assigned to one of 5 folds, and every out-of-fold score below (row-level and word-level) uses these same folds. That way no level-1 score for a training row ever comes from a model that saw that row's article.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
train_ids = train["id"].unique()
fold_of_id = pd.Series(np.arange(len(train_ids)) % N_FOLDS, index=rng.permutation(train_ids))
train_fold = train["id"].map(fold_of_id).to_numpy()


def folds(fold_labels):
    # (fit_idx, pred_idx) for each fold
    for k in range(N_FOLDS):
        yield np.flatnonzero(fold_labels != k), np.flatnonzero(fold_labels == k)


def take(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def oof_scores(make_model, X_train, y_train, X_others, fold_labels=train_fold):
    # Out-of-fold scores on train + scores on each of X_others from a model fit on all of train
    oof = np.zeros(len(y_train))
    for fit_idx, pred_idx in folds(fold_labels):
        model = make_model().fit(take(X_train, fit_idx), y_train[fit_idx])
        oof[pred_idx] = model.predict_proba(take(X_train, pred_idx))[:, 1]
    full = make_model().fit(X_train, y_train)
    return oof, [full.predict_proba(X)[:, 1] for X in X_others], full


def choose_C(make_model, X_train, y_train, candidates, fold_labels=train_fold):
    # Pick the regularisation strength on one grouped fold of train (validation stays untouched)
    fit_idx, pred_idx = next(folds(fold_labels))
    aucs = {}
    for C in candidates:
        model = make_model(C).fit(take(X_train, fit_idx), y_train[fit_idx])
        aucs[C] = roc_auc_score(y_train[pred_idx], model.predict_proba(take(X_train, pred_idx))[:, 1])
    print("  CV AUC by C:", {c: round(a, 4) for c, a in aucs.items()})
    return max(aucs, key=aucs.get)


def text_aucs(df, score):
    # AUC vs text_fake overall, and for each caption manipulation against unchanged captions
    score = pd.Series(np.asarray(score), index=df.index)
    out = {"AUC text_fake": roc_auc_score(df["text_fake"], score)}
    for m in ["text_attribute", "text_swap"]:
        mask = df["text_manipulation"].isin(["none", m])
        out[f"AUC {m} vs none"] = roc_auc_score(df.loc[mask, "text_manipulation"] == m, score[mask])
    return out


def make_logreg(C=1.0):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))


y_train = train["text_fake"].to_numpy()

## 3. Baseline: Logistic Regression on Hand-Crafted Features

The simplest model: the 19 `txt_*` features from `03` (artifact features excluded) in a scaled logistic regression. Every later model has to beat this.

In [ ]:
baseline = make_logreg().fit(train[TXT], y_train)
baseline_val = baseline.predict_proba(val[TXT])[:, 1]

results = {"Baseline: logistic regression on txt_*": text_aucs(val, baseline_val)}
pd.DataFrame(results).T.round(3)

## 4. Level 1: TF-IDF n-grams

- Word 1–2 grams: specific words and short phrases, e.g. a flipped verb
- Character 3–5 grams (`char_wb`, within word boundaries): word forms and fragments, robust to rare words
- `sublinear_tf` dampens repeated terms. `min_df` drops n-grams seen in only a few captions, which would otherwise memorise individual articles.

The vectoriser lowercases internally. The stored captions are not changed, so `fake_text_pos` stays aligned.

In [ ]:
def make_tfidf_model(C=1.0):
    ngrams = make_union(
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, sublinear_tf=True, max_features=200_000),
    )
    return make_pipeline(ngrams, LogisticRegression(C=C, max_iter=3000))


start = time.time()
C_tfidf = choose_C(make_tfidf_model, train["text"], y_train, candidates=(0.3, 1.0, 3.0))
oof, (val_s, test_s), tfidf_model = oof_scores(lambda: make_tfidf_model(C_tfidf), train["text"], y_train,
                                               [data["validation"]["text"], data["test"]["text"]])
train["l1_tfidf"], val["l1_tfidf"], data["test"]["l1_tfidf"] = oof, val_s, test_s
print(f"TF-IDF (C={C_tfidf}): {time.time() - start:.0f}s | out-of-fold AUC {roc_auc_score(y_train, oof):.3f}")

results["TF-IDF logistic regression"] = text_aucs(val, val["l1_tfidf"])
pd.DataFrame(results).T.round(3)

In [ ]:
# Which n-grams push a caption towards "manipulated"? Coefficients of the model fit on all of train_sample
names = tfidf_model[0].get_feature_names_out()
coef = pd.Series(tfidf_model[-1].coef_[0], index=names)
pd.DataFrame({
    "most 'manipulated'": coef.nlargest(20).index,
    "weight": coef.nlargest(20).values.round(2),
    "most 'genuine'": coef.nsmallest(20).index,
    "weight ": coef.nsmallest(20).values.round(2),
})

The strongest "manipulated" n-grams are ✏️[FILL: mostly sentiment words such as ... / mostly person names such as ... / a mix]. ✏️[FILL: if names dominate: the model partly learns which people appear in swapped captions, which does not generalise to new articles, and this is noted as a limitation. Otherwise: the model relies on wording rather than on who appears in the caption.]

## 5. Level 1: Word-Level Classifier

### 5.1 Word Features

One row per word of each distinct caption (about 55k captions, around 900k words), with VADER valence, sentiment contrast with the rest of the caption, GPT-2 surprise of the word, part of speech and position. The code lives in `src/text_word_features.py`, following the pattern of `src/image_features.py`.

These features are fixed functions of the caption (a sentiment lexicon, a frozen GPT-2 and spaCy tags), so nothing is fitted and computing them on all splits at once does not leak. They are computed here rather than in `03` because the table has one row per *word*, not per `row_id` like every `03` group. It is intermediate data for the word classifier below, the same role the per-face CLIP embeddings play for the face classifier in `05` (cached by `04` in `data/interim/`, not saved as a feature group). Anything *learned* from them (word frequency, the word classifier) is fitted on train only, below.

The GPT-2 pass is slow on a laptop, so the table is computed once and cached in `data/interim/text_words.parquet`. Later runs load the cache; delete the file to recompute.

In [ ]:
WORDS_PATH = PROJECT_ROOT / "data" / "interim" / "text_words.parquet"

all_rows = pd.concat([data[s] for s in SPLITS], ignore_index=True)
captions = pd.Series(all_rows["text"].unique())

if WORDS_PATH.exists():
    words = pd.read_parquet(WORDS_PATH)
else:
    import spacy
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from transformers.utils import logging as hf_logging
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    from src.text_word_features import word_table

    hf_logging.disable_progress_bar()
    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    lm_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    lm = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])

    start = time.time()
    words = word_table(captions, SentimentIntensityAnalyzer().lexicon, nlp, lm, lm_tokenizer, device)
    WORDS_PATH.parent.mkdir(parents=True, exist_ok=True)
    words.to_parquet(WORDS_PATH, index=False)
    print(f"word table computed on {device} in {(time.time() - start) / 60:.1f} min")
    del lm

assert words.groupby("text").size().reindex(captions).eq(captions.str.split().str.len().values).all(), \
    "text_words.parquet does not match the captions; delete it and re-run this cell"
print(f"{len(captions):,} captions, {len(words):,} words")

**One learned feature:** how common the word is in `train_sample` captions. Edited words are often unusual for their context, but word frequency has to be counted on train only, so it is added here and not in the module.

In [ ]:
word_counts = pd.Series(train["text"].unique()).str.lower().str.split().explode().value_counts()
words["w_log_train_freq"] = np.log1p(words["word"].str.lower().map(word_counts).fillna(0))

WORD_FEATURES = [c for c in words.columns if c.startswith("w_")]
print(len(WORD_FEATURES), "word features:", WORD_FEATURES)

### 5.2 Labels and Folds

A word is labelled 1 if its index is in `fake_text_pos` of a `text_attribute` caption. Only unchanged and text_attribute captions are used for training (see section 1). Captions shared by several rows (an image-only fake keeps the original caption) are used once.

Each training caption takes the fold of its article, so the out-of-fold word scores line up with the row-level folds of section 2.

In [ ]:
word_train = (
    train[train["text_manipulation"].isin(["none", "text_attribute"])]
    .drop_duplicates("text")[["text", "id", "fake_text_pos"]]
)
edited_words = (
    word_train[["text", "fake_text_pos"]].explode("fake_text_pos").dropna()
    .rename(columns={"fake_text_pos": "word_idx"}).astype({"word_idx": int}).assign(word_fake=1)
)
wt = (
    words.merge(word_train[["text", "id"]], on="text")
    .merge(edited_words, on=["text", "word_idx"], how="left")
    .fillna({"word_fake": 0})
)
wt["word_fake"] = wt["word_fake"].astype(int)
wt_fold = wt["id"].map(fold_of_id).to_numpy()

print(f"{wt['text'].nunique():,} training captions, {len(wt):,} words, {wt['word_fake'].sum():,} edited words")

### 5.3 Fit and Score Every Word

- Train captions: out-of-fold scores (each word is scored by a model that never saw its article)
- Captions that appear only in validation or test: a model fit on all training words
- Train text_swap captions are not in the word training set; they are scored by the model of their article's fold, so they are out-of-fold as well

In [ ]:
def make_word_gbm():
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=400, max_leaf_nodes=31, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30, random_state=RANDOM_STATE,
    )


start = time.time()
caption_fold = words["text"].map(train.drop_duplicates("text").set_index("text")["id"]).map(fold_of_id)
words["p_word"] = np.nan

for k, (fit_idx, _) in enumerate(folds(wt_fold)):
    model = make_word_gbm().fit(wt.iloc[fit_idx][WORD_FEATURES], wt.iloc[fit_idx]["word_fake"])
    in_fold = (caption_fold == k).to_numpy()
    words.loc[in_fold, "p_word"] = model.predict_proba(words.loc[in_fold, WORD_FEATURES])[:, 1]

word_model = make_word_gbm().fit(wt[WORD_FEATURES], wt["word_fake"])
not_train = words["p_word"].isna().to_numpy()
words.loc[not_train, "p_word"] = word_model.predict_proba(words.loc[not_train, WORD_FEATURES])[:, 1]
print(f"word classifier: {time.time() - start:.0f}s")

wt_scored = wt[["text", "word_idx", "word_fake"]].merge(words[["text", "word_idx", "p_word"]], on=["text", "word_idx"])
print(f"out-of-fold word-level AUC (edited vs other words): {roc_auc_score(wt_scored['word_fake'], wt_scored['p_word']):.3f}")

### 5.4 Caption Scores

Each caption is summarised by:
- `l1_word_max`: its most suspicious word
- `l1_word_mean`: the average word
- `l1_word_gap`: how far the most suspicious word stands out from the second most suspicious. One flipped word should stand out from the rest of the caption.

In [ ]:
ranked = words.sort_values(["text", "p_word"], ascending=[True, False])
ranked["rank"] = ranked.groupby("text").cumcount()
top2 = ranked[ranked["rank"] < 2].pivot(index="text", columns="rank", values="p_word")

caption_scores = pd.DataFrame({
    "l1_word_max": top2[0],
    "l1_word_mean": words.groupby("text")["p_word"].mean(),
    "l1_word_gap": top2[0] - top2[1],
})
WORD_L1 = list(caption_scores.columns)
for split in SPLITS:
    for col in WORD_L1:
        data[split][col] = data[split]["text"].map(caption_scores[col]).to_numpy()

results["Word-level: most suspicious word"] = text_aucs(val, val["l1_word_max"])
pd.DataFrame(results).T.round(3)

### 5.5 Grounding: Is the Most Suspicious Word the Edited One?

On validation text_attribute captions: the share where the top-scoring word is one of the edited words (precision@1). The baseline is picking a random word.

In [ ]:
val_attr = val[val["text_manipulation"] == "text_attribute"].drop_duplicates("text")
top_word = ranked[ranked["rank"] == 0].set_index("text")["word_idx"]
hit = np.array([top_word[t] in set(pos) for t, pos in zip(val_attr["text"], val_attr["fake_text_pos"])])
random_hit = (val_attr["fake_text_pos"].str.len() / val_attr["word_count"]).mean()

print(f"precision@1 on {len(val_attr):,} validation text_attribute captions: {hit.mean():.3f} (random word: {random_hit:.3f})")

In [ ]:
# Examples: [brackets] = most suspicious word, *stars* = the actually edited word(s)
def mark(caption, top, edited):
    out = []
    for i, w in enumerate(caption.split()):
        w = f"*{w}*" if i in edited else w
        out.append(f"[{w}]" if i == top else w)
    return " ".join(out)


for _, r in val_attr.sample(10, random_state=RANDOM_STATE).iterrows():
    print(mark(r["text"], top_word[r["text"]], set(r["fake_text_pos"])))

**Findings:** ✏️[FILL: precision@1 vs the random-word baseline, and what the examples show, e.g. the flipped verb or adjective is usually ranked first, misses tend to be ...].

## 6. Level 1: CLIP Caption Embedding (text only)

A scaled logistic regression on the CLIP caption embedding saved by `04` (the 768 `clip_txt_*` columns). CLIP’s text encoder was pretrained to match captions with images, so the embedding summarises what the caption describes as a whole, which complements the word-level signals above. Only the caption half of `clip_emb` is loaded and the embedding is computed from the caption alone, so the model stays single-mode. The regularisation strength is chosen on one grouped training fold, as for TF-IDF.

In [ ]:
start = time.time()
X_clip = {s: data[s][CLIP_TXT].to_numpy(np.float32) for s in SPLITS}
C_clip = choose_C(make_logreg, X_clip["train_sample"], y_train, candidates=(0.001, 0.01, 0.1))
oof, (val_s, test_s), _ = oof_scores(lambda: make_logreg(C_clip), X_clip["train_sample"], y_train,
                                     [X_clip["validation"], X_clip["test"]])
train["l1_clip_caption"], val["l1_clip_caption"], data["test"]["l1_clip_caption"] = oof, val_s, test_s
CLIP_L1 = ["l1_clip_caption"]
print(f"CLIP caption probe (C={C_clip}): {time.time() - start:.0f}s | out-of-fold AUC {roc_auc_score(y_train, oof):.3f}")
results["CLIP caption embedding, logistic regression"] = text_aucs(val, val["l1_clip_caption"])

pd.DataFrame(results).T.round(3)

## 7. Level 2: Gradient Boosting

Combines the level-1 scores with the hand-crafted `txt_*` features. Gradient boosting can learn interactions such as "a strongly charged word *and* high GPT-2 surprise".

In [ ]:
L1 = ["l1_tfidf"] + WORD_L1 + CLIP_L1
FINAL = TXT + L1


def make_gbm():
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=600, max_leaf_nodes=31, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30, random_state=RANDOM_STATE,
    )


def fit_level2(features):
    model = make_gbm().fit(train[features], y_train)
    return model, model.predict_proba(val[features])[:, 1]


start = time.time()
final_model, val_score = fit_level2(FINAL)
print(f"level 2 fit: {time.time() - start:.0f}s")
results["Final: gradient boosting, all features"] = text_aucs(val, val_score)
pd.DataFrame(results).T.round(3)

## 8. Ablations

What does each part add? All rows are gradient boosting on validation, except the baseline in section 3.
- **With artifact features**: how much would the model gain from caption length, which is a DGM4 construction shortcut (`03`)? A gain here is *not* a real improvement.
- **Without CLIP caption**: the purely classical NLP model.

In [ ]:
ARTIFACTS = [c for c in ARTIFACT_FEATURES if c in train.columns]
experiments = {
    "GBM: txt_* only": TXT,
    "GBM: txt_* + TF-IDF": TXT + ["l1_tfidf"],
    "GBM: txt_* + word-level": TXT + WORD_L1,
    "GBM: txt_* + TF-IDF + word-level (no CLIP)": TXT + ["l1_tfidf"] + WORD_L1,
    "GBM: final + artifact features": FINAL + ARTIFACTS,
}

start = time.time()
ablation = dict(results)
for name, features in experiments.items():
    ablation[name] = text_aucs(val, fit_level2(features)[1])
print(f"{time.time() - start:.0f}s")

ablation = pd.DataFrame(ablation).T.round(3)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = ablation.drop(index=["GBM: final + artifact features"]).reset_index(names="model").melt(
    id_vars="model", var_name="metric", value_name="AUC")
sns.barplot(data=plot_df, y="model", x="AUC", hue="metric", ax=ax)
ax.set_xlim(0.5, 1.0)
ax.set_title("Validation AUC by Model (text only)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Findings:** ✏️[FILL: what each component adds over `txt_*` alone, especially for text_attribute vs none; whether CLIP changes the result; how much the artifact features would add (a shortcut, not a real gain)].

## 9. Threshold and Validation Results

The threshold is set to maximise F1 on validation. AUC, PR-AUC and EER do not depend on it.

In [ ]:
threshold = best_f1_threshold(val["text_fake"], val_score)
pd.Series(binary_metrics(val["text_fake"], val_score, threshold)).round(3).to_frame("validation")

In [ ]:
# Share of each class flagged as caption-manipulated. For orig and the image-only classes (face_swap,
# face_attribute) the caption is untouched, so this is the false-positive rate.
detection_rate_by_type(val, val_score, threshold).round(3)

### 9.1 Operating Point: 1% False-Positive Rate

The F1 threshold suits validation's high share of fakes but flags many genuine captions. A platform would instead cap false alarms, so the threshold is also set on validation so that **1% of genuine captions** (`text_fake = 0`) are flagged, and the detection rate is reported per caption manipulation type at that threshold.

In [ ]:
TARGET_FPR = 0.01
genuine_scores = val_score[val["text_fake"].to_numpy() == 0]
threshold_1pct = float(np.quantile(genuine_scores, 1 - TARGET_FPR))
print(f"threshold for {TARGET_FPR:.0%} false-positive rate: {threshold_1pct:.3f}")

# "none" row = false-positive rate (1% by construction)
detection_rate_by_type(val, val_score, threshold_1pct, group_col="text_manipulation").round(3)

### 9.2 Robustness: Excluding Articles Also Seen in Training

Two kinds of overlap between `train_sample` and validation could inflate the scores of a model that memorises wording, which TF-IDF can do:
- **Near-duplicate photos** (`04`, section 4): validation articles whose photo has a match in `train_sample` with CLIP similarity ≥ 0.95. These are usually the same story scraped twice, so the caption often recurs too. Same rule as `05`.
- **Identical captions** (`02`, section 3): validation captions that also appear in `train_sample`.

If AUC barely changes once both are removed, the model is not relying on memorised captions.

In [ ]:
near_dups = pd.read_parquet(CLIP_CACHE / "near_duplicates.parquet")
dup_ids = set(near_dups.loc[near_dups["max_sim_to_train_sample"] >= 0.95, "id"])


def overlap_mask(df):
    """True for rows from a near-duplicate article or with a caption that also appears in train_sample."""
    return (df["id"].isin(dup_ids) | df["text"].isin(set(train["text"]))).to_numpy()


val_overlap = overlap_mask(val)
print(f"excluded {val_overlap.sum():,} validation rows ({val_overlap.mean():.1%})")
pd.DataFrame({
    "all rows": text_aucs(val, val_score),
    "excluding overlap": text_aucs(val[~val_overlap], val_score[~val_overlap]),
}).round(3)

**Findings:** ✏️[FILL: how many validation rows are excluded, and whether AUC changes. If it barely changes, the model is not relying on memorised captions].

## 10. Error Analysis

In [ ]:
# Which features does the final model rely on? Permutation importance: AUC drop on validation when a feature is shuffled.
sample = val.sample(8000, random_state=RANDOM_STATE)
r = permutation_importance(final_model, sample[FINAL], sample["text_fake"], scoring="roc_auc",
                           n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.Series(r.importances_mean, index=FINAL).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
importance.tail(15).plot.barh(ax=ax)
ax.set_title("Top features for text_fake")
ax.set_xlabel("AUC drop when shuffled")
plt.tight_layout()
plt.show()

In [ ]:
# Missed text_attribute captions (lowest scores) and genuine captions flagged with the highest scores
errors = val.assign(score=val_score).drop_duplicates("text")
with pd.option_context("display.max_colwidth", None):
    print("Missed text_attribute captions:")
    display(errors[errors["text_manipulation"] == "text_attribute"].nsmallest(8, "score")[["text", "score"]])
    print("Genuine captions with the highest scores:")
    display(errors[errors["text_manipulation"] == "none"].nlargest(8, "score")[["text", "score"]])

**Findings:** ✏️[FILL: which features rank highest in permutation importance, and what the missed text_attribute captions and the highest-scored genuine captions have in common, e.g. mild sentiment words, names, unusual phrasing].

## 11. Save Predictions

Predictions are saved per `row_id`, so the late fusion with the image model can join them directly:
- `text_model_validation.parquet`, `text_model_test.parquet`: `row_id`, `p_text_fake`
- `text_model_train_sample_oof.parquet`: **out-of-fold** scores on `train_sample`, for any model that stacks on top of this one. The level-1 inputs are already out-of-fold, so refitting level 2 per fold keeps the whole chain leak-free.
- `text_model_thresholds.csv`: the F1 and 1%-false-positive thresholds, both chosen on validation

Test predictions are written here so that later notebooks can use them without re-running this one. This step only applies the fixed model to the test captions; no test label is read until section 12.

In [ ]:
def predictions_frame(df, score):
    return pd.DataFrame({"row_id": df["row_id"].to_numpy(), "p_text_fake": score})


oof_final, _, _ = oof_scores(make_gbm, train[FINAL], y_train, [])
print(f"out-of-fold AUC of the final model on train_sample: {roc_auc_score(y_train, oof_final):.3f}")

predictions_frame(train, oof_final).to_parquet(PREDICTIONS_DIR / "text_model_train_sample_oof.parquet", index=False)
predictions_frame(val, val_score).to_parquet(PREDICTIONS_DIR / "text_model_validation.parquet", index=False)

test_score = final_model.predict_proba(data["test"][FINAL])[:, 1]  # scores only; no test labels used
predictions_frame(data["test"], test_score).to_parquet(PREDICTIONS_DIR / "text_model_test.parquet", index=False)

pd.Series({"text_fake": threshold, "text_fake_fpr1": threshold_1pct}, name="threshold").to_frame().to_csv(
    PREDICTIONS_DIR / "text_model_thresholds.csv")
print("saved to", PREDICTIONS_DIR.relative_to(PROJECT_ROOT))

## 12. Test Evaluation (run once, at the end)

Uses the model and both thresholds fixed on validation, and repeats the overlap check of section 9.2. Keep `RUN_TEST = False` until the model is final.

In [ ]:
if RUN_TEST:
    test = data["test"]  # test_score was computed in section 11 with the fixed model
    display(pd.Series(binary_metrics(test["text_fake"], test_score, threshold)).round(3).to_frame("test"))
    test_overlap = overlap_mask(test)
    display(pd.DataFrame({
        "all rows": text_aucs(test, test_score),
        "excluding overlap": text_aucs(test[~test_overlap], test_score[~test_overlap]),
    }).round(3))
    display(detection_rate_by_type(test, test_score, threshold).round(3))
    print(f"At the {TARGET_FPR:.0%}-false-positive threshold from validation:")
    display(detection_rate_by_type(test, test_score, threshold_1pct, group_col="text_manipulation").round(3))
else:
    print("RUN_TEST is False: test set not evaluated.")

# Summary

**Validation results (final model, threshold chosen on validation by F1)**

| Target | AUC | PR-AUC | EER | F1 |
|---|---|---|---|---|
| `text_fake` | ✏️[FILL] | ✏️[FILL] | ✏️[FILL] | ✏️[FILL] |

AUC by caption manipulation: text_attribute vs none ✏️[FILL], text_swap vs none ✏️[FILL].

**What the ablations show**
- ✏️[FILL: gain from TF-IDF, word level and CLIP over `txt_*` alone]
- ✏️[FILL: which manipulation the model catches, and why text_swap stays hard without the image]

**Grounding:** ✏️[FILL: precision@1 vs random word]

**Checks**
- Overlap with training (9.2): ✏️[FILL: AUC change]
- Artifact features: ✏️[FILL: gain if included; excluded from the final model]

**Operating point:** at a 1% false-positive rate, ✏️[FILL]% of text_attribute and ✏️[FILL]% of text_swap captions are flagged.

**Limitations:** ✏️[FILL: e.g. names among top n-grams, text_swap largely undetectable from text alone]